# Sample Plateau 30 DAs

This notebook inspects and reruns the clean local example package under `examples/sample_plateau_30das/`.

It uses only the packaged raw subset inside this repository.

The workflow cell below uses `overwrite=True` for the final run bundle so reruns replace any stale partial outputs.


In [1]:
from pathlib import Path
import importlib
import json

import pandas as pd

import synthetic_population_qc.context_tables as _context_tables
import synthetic_population_qc.explore.workflow as _explore_workflow
import synthetic_population_qc.synth.workflow as _synth_workflow
from synthetic_population_qc import (
    RawDataRoots,
    WorkflowSettings,
    build_base_population_cache,
    build_workflow_input_cache,
    run_energy_population_workflow,
)
from synthetic_population_qc.context_tables import load_context_tables
from synthetic_population_qc.workflow_inputs import build_workflow_input_contract, summarize_workflow_input_contract

# Force the notebook to pick up local source edits when the kernel stays alive.
importlib.reload(_context_tables)
importlib.reload(_explore_workflow)
importlib.reload(_synth_workflow)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "examples/sample_plateau_30das").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate the repository root from {start}")


c:\Users\m_hamsai\miniconda3\envs\synthetic-population-qc\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
repo_root = find_repo_root(Path.cwd())
sample_root = repo_root / "examples" / "sample_plateau_30das"
raw_root = sample_root / "raw"
processed_cache_root = sample_root / "processed_input_cache"
base_cache_root = sample_root / "base_population_cache"
run_root = sample_root / "workflow_run"
da_codes = [line.strip() for line in (sample_root / "da_codes.txt").read_text(encoding="utf-8").splitlines() if line.strip()]

raw_data = RawDataRoots.from_paths(
    data_root=sample_root,
    census_pumf_root=raw_root / "PUMF",
)

print("repo_root:", repo_root)
print("sample_root:", sample_root)
print("raw_root exists:", raw_root.exists())
print("processed_cache_root exists:", processed_cache_root.exists())
print("base_cache_root exists:", base_cache_root.exists())
print("run_root exists:", run_root.exists())
print("n_das:", len(da_codes))


repo_root: c:\Users\m_hamsai\OneDrive - Concordia University - Canada\PhD Research\Codes and Projects\synthetic-population-qc
sample_root: c:\Users\m_hamsai\OneDrive - Concordia University - Canada\PhD Research\Codes and Projects\synthetic-population-qc\examples\sample_plateau_30das
raw_root exists: True
processed_cache_root exists: True
base_cache_root exists: True
run_root exists: True
n_das: 30


In [3]:
contract = build_workflow_input_contract(
    data_root=sample_root,
    census_pumf_root=raw_root / "PUMF",
)

print(contract)
display(summarize_workflow_input_contract(
    data_root=sample_root,
    census_pumf_root=raw_root / "PUMF",
))


WorkflowInputContract(data_root=WindowsPath('c:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/synthetic-population-qc/examples/sample_plateau_30das'), census_pumf_root=WindowsPath('c:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/synthetic-population-qc/examples/sample_plateau_30das/raw/PUMF'), housing_survey_root=None, individual_pumf_candidates=(WindowsPath('c:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/synthetic-population-qc/examples/sample_plateau_30das/raw/PUMF/ind/data_donnees_2021_ind_v2.csv'), WindowsPath('c:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/synthetic-population-qc/examples/sample_plateau_30das/raw/PUMF/data_donnees_2021_ind_v2.csv')), household_pumf_candidates=(WindowsPath('c:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/synthetic-population-qc/examples

,artifact_type,artifact_name,required,found,details
0,individual_pumf,person_seed_microdata,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
1,household_pumf,household_seed_microdata,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
2,housing_survey,chs_household_housing_seed,False,False,None
3,da_census_table,age_sex_core,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
4,da_census_table,education_detailed,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
5,da_census_table,labour_detailed,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
6,da_census_table,income_detailed,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
7,da_census_table,household_type_size_detailed,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
8,da_census_table,dwelling_characteristics,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...
9,da_census_table,housing,True,True,c:\Users\m_hamsai\OneDrive - Concordia Univers...


In [4]:
context_tables = load_context_tables(sample_root)

raw_summary = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "has_da_code": "da_code" in df.columns,
    }
    for name, df in context_tables.items()
]).sort_values("table").reset_index(drop=True)

display(raw_summary)


,table,rows,cols,has_da_code
0,age_sex_core,30,43,True
1,commute,30,33,True
2,dwelling_characteristics,30,12,True
3,education_detailed,30,6,True
4,household_type_size_detailed,30,14,True
5,housing,30,26,True
6,immigration_citizenship,30,12,True
7,income_detailed,30,14,True
8,labour_detailed,30,5,True


In [5]:
processed = build_workflow_input_cache(
    raw_data=raw_data,
    cache_dir=processed_cache_root,
    overwrite=False,
)

base_cache = build_base_population_cache(
    raw_data=raw_data,
    cache_dir=base_cache_root,
    settings=WorkflowSettings.from_paths(
        output_root=sample_root,
        da_codes=da_codes,
        show_progress=False,
        generate_exploration=False,
        overwrite=True,
    ),
    overwrite=False,
)

run = run_energy_population_workflow(
    raw_data=raw_data,
    settings=WorkflowSettings.from_paths(
        output_root=sample_root,
        run_name="workflow_run",
        da_codes=da_codes,
        show_progress=False,
        generate_exploration=True,
        overwrite=True,
    ),
    processed_cache=processed,
    base_population_cache=base_cache.base_population_parquet,
)

print("processed cache:", processed.context_dir.parent)
print("base cache:", base_cache.root)
print("run root:", run.root)


processed cache: c:\Users\m_hamsai\OneDrive - Concordia University - Canada\PhD Research\Codes and Projects\synthetic-population-qc\examples\sample_plateau_30das\processed_input_cache
base cache: c:\Users\m_hamsai\OneDrive - Concordia University - Canada\PhD Research\Codes and Projects\synthetic-population-qc\examples\sample_plateau_30das\base_population_cache
run root: c:\Users\m_hamsai\OneDrive - Concordia University - Canada\PhD Research\Codes and Projects\synthetic-population-qc\examples\sample_plateau_30das\workflow_run


In [6]:
people = pd.read_parquet(run_root / "synthesis" / "people.parquet")
households = pd.read_parquet(run_root / "synthesis" / "households.parquet")
summary_metrics = pd.read_csv(run_root / "validation" / "summary_metrics.csv")
support = pd.read_csv(run_root / "validation" / "support_classification.csv")
metadata = json.loads((run_root / "manifests" / "metadata.json").read_text(encoding="utf-8"))

print("people shape:", people.shape)
print("households shape:", households.shape)
print("metadata da_codes:", len(metadata.get("da_codes") or []))

display(people.head())
display(households.head())
display(summary_metrics.sort_values(["tvd", "attribute"], ascending=[False, True]).head(12))
display(support[["attribute", "workflow_role", "support_class", "assignment_route"]])


EmptyDataError: No columns to parse from file

In [ ]:
print("sample_root:", sample_root)
print("raw_root:", raw_root)
print("processed_cache_root:", processed_cache_root)
print("base_cache_root:", base_cache_root)
print("run_root:", run_root)

sample_files = [path for path in sample_root.rglob("*") if path.is_file()]
sample_size_mb = sum(path.stat().st_size for path in sample_files) / (1024 ** 2)
print(f"sample package size: {sample_size_mb:.2f} MB")
